In [1]:
import logging
import traceback

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import mine_hard_negatives

from sentence_transformers.cross_encoder import (
    CrossEncoder,
    CrossEncoderModelCardData,
    CrossEncoderTrainer,
    CrossEncoderTrainingArguments,
)
from sentence_transformers.cross_encoder.losses import MultipleNegativesRankingLoss
from sentence_transformers.cross_encoder.evaluation import CrossEncoderNanoBEIREvaluator
from sentence_transformers.cross_encoder.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator

d:\UC15\senac_ia_uc15_nlp_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
skipHardNegativesMining:bool = True

In [3]:
train_dataset = load_dataset("parquet", data_files="D:/UC15/senac_ia_uc15_nlp_project/data/pablo/mmarco_pt_train/train-0001-of-0066.parquet")

In [4]:
train_dataset['train'][0]

{'query': 'que fruta é nativa da Austrália',
 'positive': 'Passiflora herbertiana. Um raro maracujá nativo da Austrália. Os frutos são de casca verde, polpa branca, com uma classificação comestível desconhecida. Algumas fontes listam as frutas como comestíveis, doces e saborosas, enquanto outras listam as frutas como sendo amargas e não comestíveis.assiflora herbertiana. Um raro maracujá nativo da Austrália. Os frutos são de casca verde, polpa branca, com uma classificação comestível desconhecida. Algumas fontes listam as frutas como comestíveis, doces e saborosas, enquanto outras listam as frutas como amargas e não comestíveis.',
 'negative': 'A noz de cola é o fruto da árvore da cola, um gênero (Cola) de árvores que são nativas das florestas tropicais da África.'}

In [5]:
if(not skipHardNegativesMining):

    # Mine hard negatives using a very efficient embedding model
    embedding_model = SentenceTransformer("PORTULAN/serafim-100m-portuguese-pt-sentence-encoder-ir")
    hard_train_dataset = mine_hard_negatives(
        train_dataset,
        embedding_model,
        num_negatives=5,  # How many negatives per question-answer pair
        range_min=10,  # Skip the x most similar samples
        range_max=100,  # Consider only the x most similar samples
        max_score=0.8,  # Only consider samples with a similarity score of at most x
        absolute_margin=0.1,  # Anchor-negative similarity is at least x lower than anchor-positive similarity
        relative_margin=0.1,  # Anchor-negative similarity is at most 1-x times the anchor-positive similarity, e.g. 90%
        sampling_strategy="top",  # Sample the top negatives from the range
        batch_size=4096,  # Use a batch size of 4096 for the embedding model
        output_format="labeled-pair",  # The output format is (query, passage, label), as required by BinaryCrossEntropyLoss
        use_faiss=True,  # Using FAISS is recommended to keep memory usage low (pip install faiss-gpu or pip install faiss-cpu)
    )
    print(hard_train_dataset)
    print(hard_train_dataset[1])

In [6]:
# Load a model to train/finetune
model = CrossEncoder("PORTULAN/serafim-100m-portuguese-pt-sentence-encoder-ir") # num_labels=1 is for rerankers

# Initialize the MultipleNegativesRankingLoss
# This loss requires pairs of related texts or triplets
loss = MultipleNegativesRankingLoss(model)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11682.32it/s]
BertForSequenceClassification LOAD REPORT from: PORTULAN/serafim-100m-portuguese-pt-sentence-encoder-ir
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from sentence_transformers.cross_encoder import CrossEncoderTrainingArguments

args = CrossEncoderTrainingArguments(
    # Required parameter:
    output_dir="models/finetuned_test_v1",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=False,  # Set to True if you have a GPU that supports BF16
    # batch_sampler=BatchSamplers.NO_DUPLICATES,  # losses that use "in-batch negatives" benefit from no duplicates
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=100,
    run_name="finetuned_test_v1",  # Will be used in W&B if `wandb` is installed
)

The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


: 

In [ ]:
# Set the log level to INFO to get more information
logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO)

model_name = "PORTULAN/serafim-100m-portuguese-pt-sentence-encoder-ir"
train_batch_size = 64
num_epochs = 1
num_rand_negatives = 5  # How many random negatives should be used for each question-answer pair

# 1a. Load a model to finetune with 1b. (Optional) model card data
model = CrossEncoder(
    model_name,
    model_card_data=CrossEncoderModelCardData(
        language="pt",
        license="apache-2.0",
        model_name="Serafim-100M Portuguese PT Sentence Encoder IR trained on GooAQ",
    ),
)
print("Model max length:", model.max_length)
print("Model num labels:", model.num_labels)

# 2. Load the GooAQ dataset: https://huggingface.co/datasets/sentence-transformers/gooaq
logging.info("Read the gooaq training dataset")
# full_dataset = load_dataset("sentence-transformers/gooaq", split="train").select(range(100_000))
dataset_dict = train_dataset["train"].train_test_split(test_size=1_000, seed=12)

print(dataset_dict)

train_dataset = dataset_dict["train"]
eval_dataset = dataset_dict["test"]
logging.info(train_dataset)
logging.info(eval_dataset)

# 3. Define our training loss.
loss = CachedMultipleNegativesRankingLoss(
    model=model,
    num_negatives=num_rand_negatives,
    mini_batch_size=32,  # Informs the memory usage
)

# 4. Use CrossEncoderNanoBEIREvaluator, a light-weight evaluator for English reranking
# evaluator = CrossEncoderNanoBEIREvaluator(
#     dataset_names=["msmarco", "nfcorpus", "nq"],
#     batch_size=train_batch_size,
# )
# evaluator(model)

reranking_evaluator = CrossEncoderRerankingEvaluator(
    samples=[
        {
            "query": sample["query"],
            "positive": [sample["positive"]],
            # "documents": [sample[column_name] for column_name in hard_eval_dataset.column_names[2:]],
            "negatives": [sample["negative"]],
        }
        for sample in eval_dataset
    ],
    batch_size=32,
    name="gooaq-dev",
)

# 5. Define the training arguments
short_model_name = model_name if "/" not in model_name else model_name.split("/")[-1]
run_name = f"reranker-{short_model_name}-gooaq-cmnrl"
args = CrossEncoderTrainingArguments(
    # Required parameter:
    output_dir=f"models/{run_name}",
    # Optional training parameters:
    num_train_epochs=num_epochs,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=train_batch_size,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=False,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=False,  # Set to True if you have a GPU that supports BF16
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=50,
    logging_first_step=True,
    run_name=run_name,  # Will be used in W&B if `wandb` is installed
    seed=12,
)

# 6. Create the trainer & start training
trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    evaluator=reranking_evaluator,
)
trainer.train()

# 7. Evaluate the final model, useful to include these in the model card
# evaluator(model)
results = reranking_evaluator(model)

# 8. Save the final model
final_output_dir = f"models/{run_name}/final"
model.save_pretrained(final_output_dir)

# 9. (Optional) save the model to the Hugging Face Hub!
# It is recommended to run `huggingface-cli login` to log into your Hugging Face account first
try:
    model.push_to_hub(run_name)
except Exception:
    logging.error(
        f"Error uploading model to the Hugging Face Hub:\n{traceback.format_exc()}To upload it manually, you can run "
        f"`huggingface-cli login`, followed by loading the model using `model = CrossEncoder({final_output_dir!r})` "
        f"and saving it using `model.push_to_hub('{run_name}')`."
    )

2026-03-30 21:15:42 - HTTP Request: HEAD https://huggingface.co/PORTULAN/serafim-100m-portuguese-pt-sentence-encoder-ir/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-30 21:15:42 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/PORTULAN/serafim-100m-portuguese-pt-sentence-encoder-ir/f27c45d197ea6541dd071b1d992ec91776ee76bd/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15623.44it/s]
BertForSequenceClassification LOAD REPORT from: PORTULAN/serafim-100m-portuguese-pt-sentence-encoder-ir
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-03-30 21:15:43 - HTTP Request: HEAD https://huggingface.co/PORTULAN/serafim-100m-portuguese-pt-sentence-encoder-ir/resolve/main/config.json "HTTP/1.1 307 Temporary Red

Model max length: 512
Model num labels: 1
DatasetDict({
    train: Dataset({
        features: ['query', 'positive', 'negative'],
        num_rows: 592744
    })
    test: Dataset({
        features: ['query', 'positive', 'negative'],
        num_rows: 1000
    })
})


d:\UC15\senac_ia_uc15_nlp_project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
